# 🌦️ Notebook 05: Historical Weather & Atmospheric Risk Intelligence

**RoadRisk AI Portfolio Project — Module G (Phase 7)**

This notebook demonstrates:
1. **Macro Weather-Severity Correlation**: Analyzing accident distributions across weather conditions, ambient temperatures, and visibility levels.
2. **Open-Meteo Historical API Integration**: Fetching hourly reanalysis atmospheric records for exact crash coordinates and dates.
3. **SQLite Spatiotemporal Caching**: Efficient offline disk caching with ~1km spatial grid resolution.
4. **Incident Atmospheric Re-enrichment**: Matching real historical crash records with atmospheric observations.
5. **Methodological Governance**: Exposing the "Exposure Bias" and Simpson's paradox in road accident analytics.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.weather_client import (
    fetch_historical_weather,
    get_incident_weather_match,
    compute_weather_risk_metrics,
    get_wmo_metadata,
    WMO_CODE_MAP,
    init_weather_cache
)
from src.config import DATABASE_PATH, PROCESSED_DIR

print("Libraries successfully imported!")

## 1. Load Processed Accident Data & Compute Macro Metrics

In [ ]:
data_path = PROCESSED_DIR / "indian_roads_cleaned.csv"
df_accidents = pd.read_csv(data_path)
print(f"Loaded {len(df_accidents):,} accident records.")

metrics = compute_weather_risk_metrics(df_accidents)
print(f"Total accidents: {metrics['total_accidents']:,}")
print(f"Severe relative risk ratio (Adverse vs Normal): {metrics['relative_severe_risk_ratio']:.2f}x")

## 2. Weather Condition vs Incident Severity

In [ ]:
# Cross-tabulate Weather vs Severity
cross = pd.crosstab(df_accidents["weather"], df_accidents["accident_severity"], normalize="index") * 100
print("Severity percentage by weather condition:")
print(cross.round(2))

# Adverse weather comparison breakdown
for condition, stats in metrics["adverse_comparison"].items():
    print(f"\n{condition}:")
    print(f"  Total: {stats['total_accidents']:,} ({stats['pct_of_all']}%) | Severe Rate: {stats['severe_combined_rate']}%")

## 3. Query Open-Meteo Historical Archive API with Caching

In [ ]:
# Initialize SQLite weather cache
init_weather_cache(DATABASE_PATH)

# Fetch 24-hour historical atmospheric profile for Mumbai (2024-07-15 Monsoon)
mumbai_lat, mumbai_lon = 19.0760, 72.8777
df_mumbai_weather = fetch_historical_weather(
    lat=mumbai_lat,
    lon=mumbai_lon,
    start_date="2024-07-15",
    end_date="2024-07-15",
    use_cache=True,
    db_path=DATABASE_PATH
)
print(f"Retrieved {len(df_mumbai_weather)} hourly records.")
df_mumbai_weather[["hour", "temperature_2m", "precipitation", "wmo_description", "is_adverse"]].head(10)

## 4. Single-Incident Atmospheric Re-enrichment

In [ ]:
# Select a random crash incident
sample_crash = df_accidents.sample(1, random_state=42).iloc[0]
print("Target Crash Dossier:")
print(f"ID: #{sample_crash['accident_id']} | City: {sample_crash['city']} | Date: {sample_crash['date']} {sample_crash['time']}")
print(f"Reported Weather: {sample_crash['weather']} | Temp: {sample_crash['temperature']}C | Severity: {sample_crash['accident_severity']}")

# Match exact atmospheric conditions
reanalysis = get_incident_weather_match(
    lat=sample_crash["latitude"],
    lon=sample_crash["longitude"],
    date_str=str(sample_crash["date"]),
    hour=int(sample_crash["hour"]),
    use_cache=True,
    db_path=DATABASE_PATH
)
print("\nHistorical Reanalysis Match from Open-Meteo:")
for k in ["datetime", "temperature_2m", "relative_humidity_2m", "precipitation", "wmo_description", "is_adverse"]:
    print(f"  {k}: {reanalysis.get(k)}")

## 5. Methodological Note: The Exposure Bias Fallacy

> **Caution**: In observational safety analytics, finding that more accidents occur in clear weather is driven by **traffic volume (VKT)**, not intrinsic safety. Always evaluate exposure-adjusted risk rates before deriving policy conclusions.